In [ ]:
import re

import pandas as pd

In [ ]:
def extract_title(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return " ".join(s.split()[:-2])
    elif date_re.search(s.split()[-1]):
        return " ".join(s.split()[:-1]) 
    elif s.split()[-1] in ["a", "b", "c"]:
        return " ".join(s.split()[:-1])
    else:
        return s

In [ ]:
def extract_edition(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return s.split()[-2]
    elif date_re.search(s.split()[-1]):
        return s.split()[-1] 
    elif s.split()[-1] in ["a", "b", "c"]:
        return s.split()[-1]
    else:
        return None

## Title Locations

#### title_loc_df

In [ ]:
title_loc_path = "../data/interim/title_loc_df.csv"
title_loc_df = pd.read_csv(title_loc_path, encoding="utf-8-sig", index_col=0)
assert title_loc_df.index.is_unique

# One addition for a title not in Proudfoot
title_loc_df.loc["Catechism - Penang", "entry_text"] = """
Catechism - Penang
1826
author/translator: not given
proprietor, publisher & printer: not given
Location
BL OIOC ORB.30/5553 
"""

## Bollinger

In [ ]:
BOLL_EMP_BATCH = "260625"

In [ ]:
# 1 Confirmed duplicate row for Miftah al-Bayan 1890
boll_emp_df = pd.read_csv("../data/external/boll_emp_by_date.csv", encoding="utf8", skipfooter=3, engine='python', usecols=[0, 1, 2, 3]).drop_duplicates(subset=["shelfmark", "title_edition"])

boll_emp_df["shelfmark"] = boll_emp_df["shelfmark"].str.strip()
boll_emp_df.set_index("shelfmark", inplace=True)
boll_emp_df.rename(index={"14625.d. 16": "14625.d.16"}, inplace=True)
assert boll_emp_df.index[-1] == "14625.e.5"
assert boll_emp_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

See https://github.com/harrylloyd-bl/early-malay-printed-books/issues/21 for notes on some of the changes below

In [ ]:
# modifications to boll_emp_df
# 22/06/26 have double checked all these corrections are applied to the correct cells
boll_emp_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
boll_emp_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
boll_emp_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
boll_emp_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
boll_emp_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
boll_emp_df.loc["14626.c.14(4)", "title_edition"] = "Haris Fadhillah 1900"
boll_emp_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
boll_emp_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
boll_emp_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
boll_emp_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
boll_emp_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
boll_emp_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
boll_emp_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
boll_emp_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
boll_emp_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
boll_emp_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
boll_emp_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
boll_emp_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"

In [ ]:
boll_emp_df["short_title"] = boll_emp_df["title_edition"].apply(lambda x: extract_title(x))
boll_emp_df["edition"] = boll_emp_df["title_edition"].apply(lambda x: extract_edition(x))

In [ ]:
assert boll_emp_df.set_index(["short_title"]).index.difference(title_loc_df.index).empty
assert not boll_emp_df["edition"].hasnans

In [ ]:
boll_emp_df

### Import outputs

In [ ]:
boll_emp_raw_output_df = pd.read_csv(f"../data/processed/batch_{BOLL_EMP_BATCH}/{BOLL_EMP_BATCH}_postproc.csv", encoding="utf-8-sig", index_col=[0,1]).drop(columns=["unclassified_text"])

### Fix output editions and shelfmarks

In [ ]:
# 
boll_emp_raw_output_df.rename(index={"Pelajaran Bahasa Melayu (No": "Pelajaran Bahasa Melayu (No. 1)"}, inplace=True)

boll_emp_raw_output_df.drop(index=("Terasul", "1899"), inplace=True)

boll_emp_raw_output_df.loc[("Bidayat al-Talibin", "1893"), "Shelfmark"] = "14633.d.18"
boll_emp_raw_output_df.loc[("Bidasari", "1903.a"), "Shelfmark"] = "14653.b.4"
boll_emp_raw_output_df.loc[("Hafiz al-Islam", "1899"), "Shelfmark"] = "14654.c.23"
boll_emp_raw_output_df.loc[("Ilmu Kepandaian", "n.d."), "Shelfmark"] = "14629.e.2"
boll_emp_raw_output_df.loc[("Jimak", "1891"), "Shelfmark"] = "14623.b.6"
boll_emp_raw_output_df.loc[("Kubur", "1903"), "Shelfmark"] = "14626.a.35"
boll_emp_raw_output_df.loc[("Mukhtasar al-Hikam", "1894"), "Shelfmark"] = "Jav.70"
boll_emp_raw_output_df.loc[("Salawat al-Quran", "1905"), "Shelfmark"] = "14653.b.1"
boll_emp_raw_output_df.loc[("Undang-Undang Cahaya", "1901"), "Shelfmark"] = "14622.b.13"

### Fix output shelfmarks and Citation/references note

In [ ]:
boll_emp_raw_output_df = boll_emp_raw_output_df.dropna(subset="Shelfmark").reset_index().set_index(["Shelfmark"])
boll_emp_raw_output_df.loc["14653.c.21"] = boll_emp_raw_output_df.loc["14620.i.5"]
boll_emp_raw_output_df.loc["14653.a.9"] = boll_emp_raw_output_df.loc["14625.i.8"]
boll_emp_raw_output_df.rename(index={
    "14620.ee.20": "14620.e.17",
    "14507.a.38": "14653.c.24",
    "14628.c.1(1)": "14653.d.1",
    "14620.b.33(2)": "14654.b.61",
    "14653.a.7 (10": "14653.a.7",
    "14653.a.8 (10": "14653.a.8",
    "14650.gg.93": "14650.ccc.139",
    "14653.c.25 (10": "14653.c.25",
    "14620.g.20(4)": "14626.a.37",
    
}, inplace=True)

boll_emp_raw_output_df.loc["14620.b.12(5)", "Citation/references note"] = "Proudfoot 1993: Agama Kitab a 1830s"
boll_emp_raw_output_df.loc["14625.d.9", "Citation/references note"] = "Proudfoot 1993: Ahmad Dan Muhammad 1860"

boll_emp_raw_output_df.loc["14622.b.5", "Citation/references note"] = "Proudfoot 1993: Bab al-Bai' a"

boll_emp_raw_output_df.loc["14626.c.10", "Citation/references note"] = "Proudfoot 1993: Dewa Laksana 1904"

boll_emp_raw_output_df.loc["14625.h.3", "Citation/references note"] = "Proudfoot 1993: Galila Damina a"

boll_emp_raw_output_df.loc["14629.e.2", "Citation/references note"] = "Proudfoot 1993: Ilmu Kepandaian a"

boll_emp_raw_output_df.loc["14625.h.7", "Citation/references note"] = "Proudfoot 1993: Indera Jaya 1905"
boll_emp_raw_output_df.loc["14629.a.12", "Citation/references note"] = "Proudfoot 1993: Kamus Kecil 1892"
boll_emp_raw_output_df.loc["14620.d.22(2)", "Citation/references note"] = "Proudfoot 1993: Nuh a"

boll_emp_raw_output_df.loc["14620.d.22(1)", "Citation/references note"] = "Proudfoot 1993: Pengajaran Anak Kecil a"
boll_emp_raw_output_df.loc["14620.g.13", "Citation/references note"] = "Proudfoot 1993: Rukun Sembahyang 1889"
boll_emp_raw_output_df.loc["14620.d.19(14)", "Citation/references note"] = "Proudfoot 1993: Salih a"

boll_emp_raw_output_df.loc["14629.b.16", "Citation/references note"] = "Proudfoot 1993: Terasul 1900 a"
boll_emp_raw_output_df.loc["14620.b.14(5)", "Citation/references note"] = "Proudfoot 1993: Yusuf a 1841"

In [ ]:
emp_editions_df = boll_emp_raw_output_df.join(boll_emp_df.rename_axis(index="Shelfmark"), lsuffix="_extracted", rsuffix="_emp").dropna(subset="edition_emp")
emp_editions_df.loc[emp_editions_df["Date 1"].isna(), "Date 1"] = emp_editions_df.loc[emp_editions_df["Date 1"].isna(), "year"]
emp_editions_df["Date 1"] = emp_editions_df["Date 1"].astype(int)
# emp_editions_df = pd.concat([boll_emp_raw_output_df.iloc[:2], emp_editions_df])

In [ ]:
emp_editions_df.loc[emp_editions_df["edition_extracted"] != emp_editions_df["edition_emp"].str.lower(), :].dropna(how="all", axis=1)

### Prepare for export

In [ ]:
dedup_df = emp_editions_df.reset_index().drop_duplicates(subset=["Citation/references note", "Shelfmark"]).set_index("Shelfmark")
assert dedup_df.index.sort_values().equals(boll_emp_df.index.sort_values())

boll_export_df = dedup_df.drop(columns=["short_title_extracted", "edition_extracted", "title_edition", "year", "pages", "short_title_emp", "edition_emp"]).reset_index().sort_values(by="Date 1")
# boll_export_df.to_csv(f"../data/processed/batch_{BATCH}/EMP_Bollinger_MARC_records.csv", encoding="utf-8-sig", index=False)
# boll_export_df.to_excel(f"../data/processed/batch_{BATCH}/EMP_Bollinger_MARC_records.xlsx", index=False)

## Full AAC

In [ ]:
FULL_AAC_BATCH = "260710"

See https://github.com/harrylloyd-bl/early-malay-printed-books/issues/21 for notes on the changes below

In [ ]:
full_aac_df = pd.read_csv("../data/external/full_aac_by_date.csv", encoding="utf8", usecols=[0,1,2,3]).drop_duplicates(subset=["shelfmark", "title_edition", "year"])

In [ ]:
full_aac_df["shelfmark"] = full_aac_df["shelfmark"].apply(
    lambda x: x.split(" (IOLR")[0]
).str.strip(
).str.replace("° ", ""
).str.replace(". ", "."
).str.replace(" .", "."
).str.replace("( ", "("
).str.replace("{", "("
).str.replace(".3.", ".a."
).str.replace(".33.", ".aa."
).str.replace("(1,2)", "(1),(2)"
).str.replace("(2,3,4)", "(2),(3),(4)"
).str.replace("(3,4,5)", "(3),(4),(5)")

full_aac_df.set_index("shelfmark", inplace=True)
full_aac_df.rename(index={
    "o14633.d.10": "14633.d.10",
    "14625.e,17(1)": "14625.e.17(1)",
    "l": "14626.c.16(1)",
    "14620.g.20(-)": "14620.g.20(0)",
    "14620.f.3(I)": "14620.f.3(1)",
    "I4625.g.1": "14625.g.1",
    "14653.d.1O": "14653.d.10"
}, inplace=True)

# Verified 30/06/26
full_aac_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
full_aac_df.loc["14628.c.2(3)", "title_edition"] = "Awang Sulung Merah Muda 1914"
full_aac_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
full_aac_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
full_aac_df.loc["ORB.30/5553", "title_edition"] = "Catechism - Penang"
full_aac_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
full_aac_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
full_aac_df.loc["14620.a.2", "title_edition"] = "Dini Hari 1863"
full_aac_df.loc["14620.b.33(1)", "title_edition"] = "Henry b"
full_aac_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
full_aac_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
full_aac_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
full_aac_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
full_aac_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
full_aac_df.loc["14653.d.25", "title_edition"] = "Johor 1920"
full_aac_df.loc["14620.ee.29", "title_edition"] = "Kebaktian Sehari-harian 1905"
full_aac_df.loc["14625.c.17", "title_edition"] = "Lautan Akal 1907"
full_aac_df.loc["14654.b.59(2(2))", "title_edition"] = "Madat 1919"
full_aac_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
full_aac_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
full_aac_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
full_aac_df.loc["14620.d.30(3)", "title_edition"] = "Orang Bergantung kepada Isa a"
full_aac_df.loc["14620.d.30(1)*", "title_edition"] = "Orang Miskin a"
full_aac_df.loc["14629.a.1(1)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) 1847"
full_aac_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
full_aac_df.loc["14653.d.20", "title_edition"] = "Penimbau Akal 1920"
full_aac_df.loc["14620.d.20(16)", "title_edition"] = "Perkataan Isa 1832"
full_aac_df.loc["14625.aa.2(1)", "title_edition"] = "Sabar Ali 1851"
full_aac_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
full_aac_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
full_aac_df.loc["14620.g.5*", "title_edition"] = "Targhib al-Nas 1873"
full_aac_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"
full_aac_df.loc["14622.b.15", "title_edition"] = "Undang-Undang Kapal 1916"
full_aac_df.loc["14620.c.3(3)", "title_edition"] = "Wasiat Lama 1856"
full_aac_df.loc["14625.b.1", "title_edition"] = "Yue Fei 1891"

assert full_aac_df["title_edition"].str.contains("Azimat a - fragment in binding").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Azimat a - fragment in binding", "Azimat a")

assert full_aac_df["title_edition"].str.contains("Bidasari 1903.a colophon only").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Bidasari 1903.a colophon only", "Bidasari 1903.a")

assert full_aac_df["title_edition"].str.contains("Terasul 1899 in binding only").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Terasul 1899 in binding only", "Terasul 1899")

In [ ]:
full_aac_df["short_title"] = full_aac_df["title_edition"].apply(lambda x: extract_title(x))
# apply fixes
# preceding '?' and missing spaces
full_aac_df["short_title"] = full_aac_df["short_title"].str.replace("? ", "").str.replace(
    "AbuNawas", "Abu Nawas").str.replace(
    "BarangSiapa", "Barang Siapa").str.replace(
    "CeritaJenaka", "Cerita Jenaka").str.replace(
    "KenTabuhan", "Ken Tabuhan").str.replace(
    "HajjdanUmrah", "Hajj dan Umrah").str.replace(
    "HangTuah", "Hang Tuah").str.replace(
    "MaiYoulang", "Mai Youlang").str.replace(
    "Misal HurufRumi", "Misal Huruf Rumi").str.replace(
    "NasihatBidan", "Nasihat Bidan").str.replace(
    "PenerangHati", "Penerang Hati").str.replace(
    "RajaMuda", "Raja Muda").str.replace(
    "RencanaPiatu", "Rencana Piatu")
full_aac_df["edition"] = full_aac_df["title_edition"].apply(lambda x: extract_edition(x)).str.lower()
full_aac_df.loc["ORB.30/5553", "edition"] = "1826"

In [ ]:
assert boll_emp_df.loc[boll_emp_df.index.difference(full_aac_df.index)].empty
assert full_aac_df.set_index(["short_title"]).index.difference(title_loc_df.index).empty
assert full_aac_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

In [ ]:
non_boll_aac_df = full_aac_df.set_index("title_edition", append=True).loc[full_aac_df.set_index("title_edition", append=True).index.difference(boll_emp_df.set_index("title_edition", append=True).index)].reset_index(level=1) # .set_index(["short_title", "edition", "shelfmark"])
idx_match_df = non_boll_aac_df.reset_index().set_index(["short_title", "edition", "shelfmark"])
assert len(non_boll_aac_df) + len(boll_emp_df) == len(full_aac_df)
# assert(non_boll_aac_df.index.value_counts().max() == 1)
len(non_boll_aac_df), len(boll_emp_df), len(full_aac_df)

In [ ]:
non_boll_aac_entry_df = title_loc_df.merge(non_boll_aac_df, how="inner", left_index=True, right_on="short_title").reset_index().set_index("shelfmark")

### Import outputs

In [ ]:
full_aac_raw_output_df = pd.read_csv(f"../data/processed/batch_{FULL_AAC_BATCH}/{FULL_AAC_BATCH}_postproc.csv", encoding="utf-8-sig").drop(columns=["unclassified_text"])
full_aac_raw_output_df["edition"] = full_aac_raw_output_df["edition"].str.replace(".8", ".a").str.replace("19125", "1915").str.replace("19U5", "1915").replace("1920+", "1920").str.lower()
full_aac_raw_output_df["Shelfmark"] = full_aac_raw_output_df["Shelfmark"].str.replace(" (10", "")
full_aac_raw_output_df = full_aac_raw_output_df.set_index(["short_title", "edition", "Shelfmark"])

# Remove some duplicate shelfmarks that refer to the wrong item
full_aac_raw_output_df.drop(index=[
    ("Haris Fadhillah", "1900", "14626.c.14(4)"),
    ("Bidasari", "1903.b", "14626.c.14(8)")
], inplace=True)

full_aac_raw_output_df = full_aac_raw_output_df.reset_index(level=2)

### Fix output editions and shelfmarks

In [ ]:
# These ones haven't been extracted in post-processing
full_aac_raw_output_df.loc[("Air Mawar", "1899"), "Shelfmark"] = "14626.d.11(7)"

full_aac_raw_output_df.loc[("Amin", "1863"), "Shelfmark"] = "14620.f.1"

full_aac_raw_output_df.loc[("Bible: New Testament", "1911.a"), "Shelfmark"] = "Jav.96"

full_aac_raw_output_df.loc[("Dictionary: Wilkinson", "1901-03"), "Shelfmark"] = "15012.fa.1"

full_aac_raw_output_df.loc[("Dini Hari", "1863"), "Shelfmark"] = "14620.a.2"

full_aac_raw_output_df.loc[("Jamita Kuria", "1901-02"), "Shelfmark"] = "14633.c.12"

full_aac_raw_output_df.loc[("Makan Sirih", "1919"), "Shelfmark"] = "14654.b.59(2(3))"

full_aac_raw_output_df.loc[("Paulus", "1863"), "Shelfmark"] = "14620.b.36"

full_aac_raw_output_df.loc[("Pelajaran Bahasa Melayu (No.1)", "1847"), "Shelfmark"] = "14629.a.1(1)"
full_aac_raw_output_df.loc[("Pelajaran Bahasa Melayu (No.2)", "1861"), "Shelfmark"] = "14629.a.1(2)"

full_aac_raw_output_df.loc[("Pelajaran Sekolah Sabat", "1920"), "Shelfmark"] = "14620.c.7"

full_aac_raw_output_df.loc[("Penambah Akal", "1917"), "Shelfmark"] = "14653.b.12"

full_aac_raw_output_df.loc[("Pengajaran Injil", "1863"), "Shelfmark"] = "14620.a.6"

full_aac_raw_output_df.loc[("Perang Dunia: Daniels", "1919"), "Shelfmark"] = "14620.d.5"

full_aac_raw_output_df.loc[("Perukunan", "1901"), "Shelfmark"] = "14620.g.18"

full_aac_raw_output_df.loc[("Raffles", "1919"), "Shelfmark"] = "14624.b.10"

full_aac_raw_output_df.loc[("Salawat al-Quran", "1905"), "Shelfmark"] = "14519.b.77"

full_aac_raw_output_df.loc[("Sanhe Baojian", "1910-16"), "Shelfmark"] = "14625.a.4"

full_aac_raw_output_df.loc[("Siraj al-Kalbi", "1916"), "Shelfmark"] = "14653.b.8"

full_aac_raw_output_df.loc[("Undang-Undang Kapal", "1916"), "Shelfmark"] = "14622.b.15"

In [ ]:
assert (full_aac_raw_output_df.dropna(subset="Shelfmark")[full_aac_raw_output_df.dropna(subset="Shelfmark")["Shelfmark"].str.contains(r"\(") & ~full_aac_raw_output_df.dropna(subset="Shelfmark")["Shelfmark"].str.contains(r"\)")]).empty

### Fix output shelfmarks and Citation/references note

In [ ]:
non_boll_aac_df.loc['14628.c.1(4)', "edition"] = "1907-08"
non_boll_aac_df.loc['14653.d.4(1),(2)', "edition"] = "1907-08"
non_boll_aac_df.loc['14653.d.46(1),(2)', "edition"] = "1911-15"
non_boll_aac_df.loc['14624.c.4', "edition"] = "1915-17"
non_boll_aac_df.loc['14653.d.26(2)', "edition"] = "1915-17"

idx_match_df = non_boll_aac_df.reset_index().set_index(["short_title", "edition", "shelfmark"])

In [ ]:
# check title and edition match (not shelfmark here)
captured = full_aac_raw_output_df.join(idx_match_df.reset_index(level=2), how="inner")

In [ ]:
missing = idx_match_df.index.difference(captured.index)
# I've accounted that these are OK based on shelfmarks
# Largely just editions that are stated as single years in AAC list but are ranges in Proudfoot
assert missing.shape == (34,)  

In [ ]:
[x for x in missing]

### For setting shelfmark index

In [ ]:
full_aac_sm_output_df = full_aac_raw_output_df.dropna(subset="Shelfmark").reset_index().set_index(["Shelfmark"])

In [ ]:
# Abu Nawas - date is wrong rather than missing
full_aac_sm_output_df.loc['14625.a.15', ["edition", "Date 1"]] = "1916"
full_aac_sm_output_df.loc['14625.a.15', "Citation/references note"] = "Proudfoot 1993: Abu Nawas 1916"
full_aac_sm_output_df.loc['14653.b.41', ["edition", "Date 1"]] = "1916"
full_aac_sm_output_df.loc['14653.b.41', "Citation/references note"] = "Proudfoot 1993: Abu Nawas 1916"

# 'Acara Manusia',
full_aac_sm_output_df.loc["14650.bbb.4"] = full_aac_sm_output_df.loc["14650.a.94"]

# 'Alf Lailah wa Lailah',
full_aac_sm_output_df.loc["14653.a.6"] = full_aac_sm_output_df.loc["14625.i.5"]

# 'Awang Sulung Merah Muda',
full_aac_sm_output_df.loc["14653.d.5"] = full_aac_sm_output_df.loc["14628.c.2(3)"]

# 'Bible: Luke',
full_aac_sm_output_df.loc["14653.b.26"] = full_aac_sm_output_df.loc["14620.c.20(6)"]

# 'Bible: New Testament',
full_aac_sm_output_df.loc["Jav.121"] = full_aac_sm_output_df.loc["Jav.96"]
full_aac_sm_output_df.loc["Jav.161"] = full_aac_sm_output_df.loc["Jav.96"]

full_aac_sm_output_df.loc["14620.e.14"] = full_aac_sm_output_df.loc["14620.ee.20"]

# 'Kisah-Kisah Kitab Injil',
full_aac_sm_output_df.loc["14620.b.14(3)"] = full_aac_sm_output_df.loc["14620.c.6"]

# 'Malim Dewa',
full_aac_sm_output_df.loc["14653.d.7"] = full_aac_sm_output_df.loc["14628.c.1(7)"]

# 'Perang Dunia: Daniels',
full_aac_sm_output_df.loc["14653.b.7"] = full_aac_sm_output_df.loc["14620.d.5"]

# 'Raffles',
full_aac_sm_output_df.loc["14653.b.20"] = full_aac_sm_output_df.loc["14624.b.10"]

# 'Surat Paulus',
full_aac_sm_output_df.loc["14654.b.21"] = full_aac_sm_output_df.loc["14620.b.14(6)"]

# 'Undang-Undang Kapal'
full_aac_sm_output_df.loc["14653.d.5"] = full_aac_sm_output_df.loc["14628.c.2(3)"]

In [ ]:
captured = full_aac_sm_output_df.join(non_boll_aac_df.rename_axis(index="Shelfmark"), lsuffix="_extracted", rsuffix="_emp", how="inner")
missing = non_boll_aac_df.index.difference(captured.index)
assert missing.empty

In [ ]:
full_aac_editions_df = full_aac_sm_output_df.join(non_boll_aac_df.rename_axis(index="Shelfmark"), lsuffix="_extracted", rsuffix="_emp", how="inner")
full_aac_editions_df = full_aac_editions_df.set_index(["edition_extracted", "edition_emp"], append=True)
full_aac_editions_df.drop(index=[
    ("14653.b.4", "1903.a", "1903.b"),
    ("14653.b.4", "1903.b", "1903.a"),
    ("14629.c.48(2)", "1903.b", "1904"),
    ("14629.c.48(2)", "1904", "1903.b")
], inplace=True)
full_aac_editions_df = full_aac_editions_df.reset_index(level=[1,2])

assert full_aac_editions_df.index.value_counts().sort_index().equals(non_boll_aac_df.index.value_counts().sort_index())
assert full_aac_editions_df.value_counts(subset=["Citation/references note", "Shelfmark"]).max() == 1
assert full_aac_editions_df.index.sort_values().equals(non_boll_aac_df.index.sort_values())

full_aac_editions_df.loc[full_aac_editions_df["Date 1"].isna(), "Date 1"] = full_aac_editions_df.loc[full_aac_editions_df["Date 1"].isna(), "year"]
full_aac_editions_df["Date 1"] = full_aac_editions_df["Date 1"].astype(int)
full_aac_editions_df["Date 2"] = full_aac_editions_df["Date 2"].astype(pd.Int64Dtype())

def gen_method_of_acquisition(date):
    if date <= 1886:
        method = "purchased"
    elif date >= 1887:
        method = "legal deposit"
    return method

full_aac_editions_df.loc[full_aac_editions_df["Method of acquisition"].isna(), "Method of acquisition"] = full_aac_editions_df[full_aac_editions_df["Method of acquisition"].isna()]["Date 1"].apply(lambda x: gen_method_of_acquisition(x))

In [ ]:
full_aac_editions_df.loc[full_aac_editions_df["edition_extracted"] != full_aac_editions_df["edition_emp"].str.lower(), :].iloc[::][["edition_extracted", "edition_emp"]]

### Prepare for export

In [ ]:
non_boll_aac_export_df = full_aac_editions_df.drop(columns=["short_title_extracted", "edition_extracted", "edition_emp", "title_edition", "year", "pages", "short_title_emp"]).sort_index().reset_index()
# non_boll_aac_export_df.to_csv(f"../data/processed/batch_{FULL_AAC_BATCH}/EMP_AAC_MARC_records.csv", encoding="utf-8-sig", index=False)
# non_boll_aac_export_df.to_excel(f"../data/processed/batch_{FULL_AAC_BATCH}/EMP_AAC_MARC_records.xlsx", index=False)

### Combined export to xlsx

In [ ]:
with pd.ExcelWriter("../data/processed/Proudfoot_AAC_EMP_MARC_records.xlsx") as writer:
    pd.concat([boll_export_df, non_boll_aac_export_df]).sort_index().to_excel(writer, sheet_name="Full AAC by shelfmark", index=False)
    boll_export_df.sort_index().to_excel(writer, sheet_name="Bollinger EMP by shelfmark", index=False)
    non_boll_aac_export_df.sort_index().to_excel(writer, sheet_name="Non Bollinger AAC by shelfmark", index=False)